In [14]:
import argparse
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

In [81]:

INPUT_FILE  = "jadbio_sweet.csv"   #label m balancing
TARGET_COL  = "sweet"              # the label column name
SEP         = ";"                 

TEST_SIZE   = 0.20   
SMOTE_RATIO = 0.50   
K_NEIGHBORS = 5      # number of nearest neighbors for SMOTE
RANDOM_SEED = 42

In [82]:
df = pd.read_csv(INPUT_FILE, sep=SEP)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce').fillna(0).astype(int)

feat_cols = [c for c in df.columns if c != TARGET_COL]
pos_count = int(df[TARGET_COL].sum())
neg_count = len(df) - pos_count

print(f"File         : {INPUT_FILE}")
print(f"Total rows   : {len(df)}")
print(f"Features     : {len(feat_cols)}")
print(f"Positive (1) : {pos_count} ({pos_count/len(df)*100:.1f}%)")
print(f"Negative (0) : {neg_count} ({neg_count/len(df)*100:.1f}%)")

File         : jadbio_sweet.csv
Total rows   : 4981
Features     : 1005
Positive (1) : 1928 (38.7%)
Negative (0) : 3053 (61.3%)


## Detect the binary and continous features 

In [84]:
X_all = df[feat_cols].values.astype(float)

binary_mask = np.array([
    set(np.unique(X_all[~np.isnan(X_all[:, j]), j])).issubset({0.0, 1.0})
    for j in range(X_all.shape[1])
])

print(f"Binary features (MACCS + Morgan) : {binary_mask.sum()}")
print(f"Continuous features (physico)    : {(~binary_mask).sum()}")

Binary features (MACCS + Morgan) : 216
Continuous features (physico)    : 789


## Split the data 

In [85]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(sss.split(df, df[TARGET_COL]))

train = df.iloc[train_idx].reset_index(drop=True)
test  = df.iloc[test_idx].reset_index(drop=True)

pos_tr = int(train[TARGET_COL].sum())
pos_te = int(test[TARGET_COL].sum())

print(f"Train set : {len(train)} rows | pos={pos_tr} ({pos_tr/len(train)*100:.1f}%) | neg={len(train)-pos_tr} ({(len(train)-pos_tr)/len(train)*100:.1f}%)")
print(f"Test set  : {len(test)} rows  | pos={pos_te} ({pos_te/len(test)*100:.1f}%) | neg={len(test)-pos_te} ({(len(test)-pos_te)/len(test)*100:.1f}%)")


Train set : 3984 rows | pos=1542 (38.7%) | neg=2442 (61.3%)
Test set  : 997 rows  | pos=386 (38.7%) | neg=611 (61.3%)


## SMOTE Function

katgenerer synthetic data bax nbalanciw data , l'algorithm chkaydir : 

kaya5oud k neighbors kayrsm ligne ma bin random point ou douk neighbors then  kaygenere point random fdik ligne 


In [86]:
def run_smote(X_pos, n_needed, binary_mask, k=5, seed=42):
 
    rng = np.random.default_rng(seed)
    k   = min(k, len(X_pos) - 1)
 
    # find k nearest neighbors for each positive sample
    nn = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1)
    nn.fit(X_pos)
    _, neighbors = nn.kneighbors(X_pos)  # col 0 = self, cols 1.. = neighbors
 
    synthetic = np.zeros((n_needed, X_pos.shape[1]))
    for i in range(n_needed):
        base_idx     = rng.integers(0, len(X_pos))
        neighbor_idx = neighbors[base_idx, rng.integers(1, k + 1)]
        gap          = rng.uniform(0, 1)
 
        new = X_pos[base_idx] + gap * (X_pos[neighbor_idx] - X_pos[base_idx])

        #kanroundiw lfeature li binary l  0 aw 1
        new[binary_mask] = np.round(new[binary_mask]).clip(0, 1)
        synthetic[i] = new
 
    return synthetic

## Apply the SMOTE 

In [87]:
# Isolate positive samples from training set
pos_train = train[train[TARGET_COL] == 1].reset_index(drop=True)
neg_train = train[train[TARGET_COL] == 0].reset_index(drop=True)

# How many synthetic samples do we need?
pos_target_n = int(SMOTE_RATIO * len(neg_train) / (1 - SMOTE_RATIO))
n_synthetic  = pos_target_n - len(pos_train)

print(f"Real positive samples in train : {len(pos_train)}")
print(f"Target after SMOTE             : {pos_target_n}")
print(f"Synthetic samples to generate  : {n_synthetic}")

Real positive samples in train : 1542
Target after SMOTE             : 2442
Synthetic samples to generate  : 900


In [88]:
# Impute NaN values with column median (needed for KNN distance computation)
X_pos = pos_train[feat_cols].values.astype(float)

medians  = np.nanmedian(X_pos, axis=0)
medians  = np.where(np.isnan(medians), 0.0, medians)
X_pos_knn = X_pos.copy()

nan_mask = np.isnan(X_pos)
if nan_mask.any():
    X_pos_knn[nan_mask] = np.take(medians, np.where(nan_mask)[1])
    print(f"Imputed {nan_mask.sum()} NaN values with column median")
else:
    print("No NaN values found")

Imputed 327 NaN values with column median


In [89]:
synthetic =run_smote(X_pos_knn, n_synthetic, binary_mask, k=K_NEIGHBORS, seed=RANDOM_SEED)

In [90]:
# Quality check — how different are synthetic samples from real ones?
stds  = np.nanstd(X_pos_knn, axis=0)
stds[stds == 0] = 1.0
drift = np.abs((X_pos_knn.mean(0) - synthetic.mean(0)) / stds).mean()

if   drift < 0.1: quality = "excellent"
elif drift < 0.3: quality = "good"
else:             quality = "check manually"

print(f"Normalised drift: {drift:.4f}  [{quality}]")
# The lower the drift, the closer synthetic samples are to real ones in feature space

Normalised drift: 0.0285  [excellent]


In [91]:
# Build synthetic dataframe
synth_df = pd.DataFrame(synthetic, columns=feat_cols)
synth_df[TARGET_COL] = 1

# Combine original train + synthetic, then shuffle
train_balanced = pd.concat([train, synth_df], ignore_index=True)
train_balanced = train_balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

pf = int(train_balanced[TARGET_COL].sum())
print(f"Final training set:")
print(f"  Total rows : {len(train_balanced)}")
print(f"  Positive   : {pf}  ({pf/len(train_balanced)*100:.1f}%)")
print(f"  Negative   : {len(train_balanced)-pf}  ({(len(train_balanced)-pf)/len(train_balanced)*100:.1f}%)")
print(f"  Real rows  : {len(train)}")
print(f"  Synthetic  : {n_synthetic}")

Final training set:
  Total rows : 4884
  Positive   : 2442  (50.0%)
  Negative   : 2442  (50.0%)
  Real rows  : 3984
  Synthetic  : 900


In [92]:
stem      = Path(INPUT_FILE).stem
out_train = f"{stem}_train_balanced.csv"
out_test  = f"{stem}_test_real.csv"

train_balanced.to_csv(out_train, sep=SEP, index=False)
test.to_csv(out_test,  sep=SEP, index=False)

print(f"Saved: {out_train}")

print()
print(f"Saved: {out_test}")


Saved: jadbio_sweet_train_balanced.csv

Saved: jadbio_sweet_test_real.csv
